# Praetor — SFT training notebook

This notebook trains the **Praetor** incident-response agent end-to-end:

1. **SFT** (supervised fine-tune) the policy on senior-SRE behavioral-clone
   trajectories drawn from `IDEAL_TRAJECTORIES` in `coach.py`.
2. **Evaluate** all 3 trained scenario families across random / SFT.
3. **Render** plots into `/content/results/` for the README.
4. **Push** the LoRA adapter to your HuggingFace Hub.

### Why SFT-only?

We originally planned SFT + GRPO. Across multiple Colab + HF Spaces sessions,
GRPO via Unsloth's compiled-cache runner hit a sequence of upstream version-skew
bugs (`None` ref-logprobs, `grpo_accumulated_loss` tuple-unpacking mismatch,
TRL `GRPOTrainer` import paths). Vanilla TRL's GRPOTrainer works but adds
~2 hours to training. To make the hackathon deadline reliably, this notebook
ships **SFT-only** — which already produces:

- A clean **SFT loss curve** (the "loss plot" the rubric requires)
- A **random-vs-SFT before/after comparison** (the "improvement evidence" the rubric requires)
- A **trained LoRA adapter** pushed to HuggingFace Hub

Cells 7 and 8 (GRPO dataset + GRPO training) are kept as no-ops with the
GRPO config preserved as documentation — re-enable when the upstream
version-skew is resolved.

### Stack

This notebook uses the **vanilla HuggingFace stack** (transformers + peft +
bitsandbytes + trl). Unsloth speedup was abandoned after compiled-cache
version-skew bugs blocked GRPO.

### Total runtime budget (SFT-only)

| Phase | A100 / L40S | T4 |
|---|---:|---:|
| Setup + clone | 5 min | 5 min |
| Load model (4-bit Qwen 1.5B + LoRA) | 4 min | 5 min |
| SFT (1 epoch, skipped if `sft.done` exists) | ~60 min | ~120 min |
| SFT sanity-eval (9 episodes) | ~5 min | ~8 min |
| Final eval (60 episodes) | ~8 min | ~15 min |
| Plots + push | 3–5 min | 3–5 min |
| **Total** | **~90 min** | **~155 min** |

If `sft.done` exists from a prior run, cell 5 is a no-op and total drops by
~60 min on A100/L40S. The adapter at `/content/checkpoints/sft_adapter/`
survives runtime restarts.

### Before you run

- **Runtime → Change runtime type → A100 / L40S GPU** (or T4 if unavailable).
- **Run all** from the menu, then walk away. The notebook writes intermediate
  state to `/content/checkpoints/` so a disconnect doesn't lose work.
- If any cell errors, the next cell will refuse to start. Fix the error,
  re-run the failing cell, then continue.


## Cell 1 - Setup: install dependencies + clone the repo

**Expected runtime: 2–3 minutes (any GPU).** Most of the wall-clock is the
`unsloth` install and a single git clone.


In [ ]:
import time, os, sys, subprocess
_t0 = time.monotonic()

# Vanilla HF/PEFT/TRL stack — no Unsloth.
%pip install -q --upgrade pip 2>&1 | tail -1
%pip install -q "torch>=2.4" "transformers>=4.46,<4.50" "trl==0.15.2" "peft>=0.13" "accelerate>=0.34" "bitsandbytes>=0.43" 2>&1 | tail -1
%pip install -q "datasets>=2.14" "huggingface-hub>=0.20" "matplotlib>=3.7" 2>&1 | tail -1
%pip install -q "pydantic>=2.0" "fastapi>=0.104" 2>&1 | tail -1

# Clone (or update) the project so imports like `training.eval_runner` resolve.
if not os.path.exists("/content/incident-commander"):
    !git clone -q https://github.com/root4shreshth/incident-commander.git /content/incident-commander
else:
    !cd /content/incident-commander && git pull -q
sys.path.insert(0, "/content/incident-commander")

# CRITICAL: change CWD to project root BEFORE other imports run.
# When the notebook lives at training/train_grpo.ipynb, Jupyter sets CWD
# to the training/ dir, which makes `from datasets import Dataset` (in
# cell 4) resolve to the LOCAL training/datasets.py file (which only has
# build_sft_dataset, not Dataset) instead of the HF `datasets` library.
# No-op on Colab where CWD is already /content.
os.chdir("/content/incident-commander")
print(f"CWD: {os.getcwd()}")

os.makedirs("/content/checkpoints", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

# Sanity-check: surface any pip resolution issues NOW, not 30 min into a run.
from trl import SFTTrainer, SFTConfig
print(f"Setup OK in {time.monotonic()-_t0:.1f}s. Vanilla stack, trl==0.15.2, SFT-only.")


## Cell 2 - Compute detection

**Expected runtime: < 1 second.** Reports the GPU and adjusts later cells'
batch sizes accordingly.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Runtime → Change runtime type → A100 (or T4)."
    )
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_A100 = "A100" in GPU_NAME

# Batch size + GRPO step counts scale with GPU memory.
SFT_BATCH = 4 if IS_A100 else 2
SFT_GRAD_ACCUM = 4 if IS_A100 else 8        # effective batch = 16 either way
GRPO_STEPS = 60 if IS_A100 else 30   # A100 cut to 60 for ~40-min GRPO + buffer (was 200)
GRPO_NUM_GENERATIONS = 4 if IS_A100 else 2  # rollouts per prompt
GRPO_BATCH = 2 if IS_A100 else 1

print(f"GPU            : {GPU_NAME} ({GPU_MEM_GB:.1f} GB)")
print(f"SFT batch      : {SFT_BATCH} × grad_accum {SFT_GRAD_ACCUM} (eff. {SFT_BATCH*SFT_GRAD_ACCUM})")
print(f"GRPO steps     : {GRPO_STEPS}")
print(f"GRPO rollouts  : {GRPO_NUM_GENERATIONS} per prompt")


## Cell 3 - Load Qwen2.5-Coder-1.5B in 4-bit with LoRA

**Expected runtime: 2–3 minutes on A100, 3–5 minutes on T4.** Downloads
~1.5 GB of model weights, applies the 4-bit quantization, and attaches
a LoRA r=16 adapter ready for SFT.

Skip-if-already-done: if `model` is already in scope (re-running the
notebook after a partial run), this cell is a no-op.


In [ ]:
import time, os, torch
_t0 = time.monotonic()

if "model" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

    MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if IS_A100 else torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)

    # Reuse the SFT adapter from a prior session if it survived the runtime
    # restart. /content/ is persistent in Colab. Saves ~30-60 min of retraining.
    SFT_CKPT = "/content/checkpoints/sft_adapter"
    SFT_DONE = "/content/checkpoints/sft.done"
    if os.path.exists(SFT_CKPT) and os.path.exists(SFT_DONE):
        print(f"Loading existing SFT adapter from {SFT_CKPT}")
        model = PeftModel.from_pretrained(model, SFT_CKPT, is_trainable=True)
    else:
        print("Attaching fresh LoRA r=16 (will train from scratch in cell 5)")
        lora_config = LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)

    print(f"Model loaded + LoRA attached in {time.monotonic()-_t0:.1f}s")
    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()
else:
    print("Model already loaded; skipping.")


## Cell 4 - Build the SFT chat dataset

**Expected runtime: < 30 seconds.** Materializes ~120 (system, user, assistant)
chat rows from `IDEAL_TRAJECTORIES` in `incident_commander_env/server/coach.py`,
sampled across multiple seeds per family so each scenario gets several
parametric variants.


In [ ]:
import time
_t0 = time.monotonic()

from training.datasets import build_sft_dataset, to_chat_messages, SYSTEM_PROMPT
from datasets import Dataset

raw_rows = build_sft_dataset(n_seeds_per_family=8)
hf_rows = []
for r in raw_rows:
    msgs = to_chat_messages(r)
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    hf_rows.append({"text": text})

sft_ds = Dataset.from_list(hf_rows)
print(f"SFT dataset: {len(sft_ds)} rows  ({time.monotonic()-_t0:.1f}s)")
print(f"  example length (chars): {len(sft_ds[0]['text'])}")
print(f"  scenarios covered     : {sorted({r['scenario'] for r in raw_rows})}")


## Cell 5 - SFT training (1 epoch)

**Expected runtime: 30–40 minutes on A100, 75–90 minutes on T4.** This is
the longest training cell; you'll see TRL's per-step progress bar.

Skip-if-already-done: if `/content/checkpoints/sft.done` exists from a
previous successful run, the cell loads weights from
`/content/checkpoints/sft_adapter/` instead of re-training.


In [ ]:
import os, time
_t0 = time.monotonic()

SFT_CHECKPOINT = "/content/checkpoints/sft_adapter"
SFT_DONE = "/content/checkpoints/sft.done"

if os.path.exists(SFT_DONE) and os.path.exists(SFT_CHECKPOINT):
    print("SFT already complete (sft.done exists). Adapter loaded in cell 3.")
else:
    from trl import SFTTrainer, SFTConfig

    sft_args = SFTConfig(
        output_dir="/content/sft_runs",
        per_device_train_batch_size=SFT_BATCH,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        logging_steps=2,
        save_strategy="no",
        bf16=IS_A100, fp16=not IS_A100,
        max_seq_length=2048,
        dataset_text_field="text",
        report_to="none",
        seed=42,
        gradient_checkpointing=True,
    )
    sft_trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=sft_ds, args=sft_args,
    )
    sft_trainer.train()
    model.save_pretrained(SFT_CHECKPOINT)
    tokenizer.save_pretrained(SFT_CHECKPOINT)
    open(SFT_DONE, "w").write("done")

print(f"SFT phase complete in {time.monotonic()-_t0:.1f}s")


## Cell 6 - SFT sanity-eval (9 episodes)

**Expected runtime: ~2 minutes on A100, ~5 minutes on T4.** Quick
random-vs-SFT comparison on 3 seeds × 3 families to confirm SFT learned
something before we spend hours on GRPO.

If SFT scores are below random, abort and inspect the SFT loss curve.


In [ ]:
import time, warnings, logging
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", category=FutureWarning,
                        module=r"transformers\.modeling_attn_mask_utils")
logging.getLogger("transformers").setLevel(logging.ERROR)

from training.eval_runner import evaluate, random_policy, hf_pipeline_policy
from training.datasets import SYSTEM_PROMPT
from transformers import GenerationConfig

model.eval()  # vanilla equivalent of Unsloth's FastLanguageModel.for_inference

# Clear Qwen's default max_length=32768 + use the right stop tokens.
# This is the fix that turned 90-minute hangs into 2-minute evals.
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.max_length = None
_eos_ids = [tokenizer.eos_token_id]
_im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if _im_end is not None and _im_end != tokenizer.unk_token_id and _im_end not in _eos_ids:
    _eos_ids.append(_im_end)

GEN_CFG = GenerationConfig(
    max_new_tokens=160,
    do_sample=False,
    eos_token_id=_eos_ids,
    pad_token_id=tokenizer.eos_token_id,
)

def hf_generate(messages, max_new=160):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    out = model.generate(**inputs, generation_config=GEN_CFG)
    # Decode ONLY the newly-generated tokens. Slicing on token IDs avoids
    # the bug where `full[len(text):]` fails because tokenize=False keeps
    # special tokens in `text` but skip_special_tokens=True strips them
    # from `full` - so `full.startswith(text)` is False and the whole
    # transcript (system prompt + user + assistant) gets returned, which
    # then makes the JSON parser pick up the EXAMPLE inside the system
    # prompt instead of the model's actual response.
    new_tokens = out[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

SANITY_FAMILIES = ["oom_crash", "db_pool_exhaustion", "bad_deployment_cascade"]
SANITY_SEEDS = [2000, 2001, 2002]
TOTAL = len(SANITY_FAMILIES) * len(SANITY_SEEDS)

def _progress(idx, total, family, seed, ep):
    tag = "OK" if ep.resolved else "X "
    print(f"  [{idx:>2}/{total}] [{tag}] {family:<25s} seed={seed} steps={ep.steps_used:>2} score={ep.score:.2f}", flush=True)

_t0 = time.monotonic()
print("→ random baseline")
report_random = evaluate("random", random_policy(rng_seed=99),
                         SANITY_FAMILIES, SANITY_SEEDS,
                         system_prompt=SYSTEM_PROMPT, on_episode=_progress)

print(f"\n→ SFT model")
report_sft = evaluate("sft", hf_pipeline_policy(hf_generate),
                      SANITY_FAMILIES, SANITY_SEEDS,
                      system_prompt=SYSTEM_PROMPT, on_episode=_progress)

print(f"\nSanity eval done in {time.monotonic()-_t0:.1f}s")
print("=" * 60)
for cond, rpt in [("random", report_random), ("sft", report_sft)]:
    print(f"\n  {cond}:")
    for fam, stats in rpt.by_family.items():
        print(f"    {fam:<25s}: success={stats['success_rate']*100:>4.0f}%  "
              f"score={stats['avg_score']:.2f}  steps={stats['avg_steps_used']:.1f}")


## Cell 7 — (DISABLED) Build the GRPO prompt dataset

**Disabled** in this SFT-only run. See the "Why SFT-only?" note at the top.
Cell is kept as documentation; runs as a no-op.


In [ ]:
# GRPO disabled in this SFT-only run.
#
# When re-enabling, restore the original body that builds GRPO_STEPS
# prompts via training.curriculum.Curriculum and writes them into
# `grpo_ds: datasets.Dataset`. The original code is preserved in git
# history and the GRPOConfig in cell 8 below is intact for reference.
print("[skip] Cell 7 — GRPO dataset build disabled (SFT-only run).")


## Cell 8 — (DISABLED) GRPO training

**Disabled** in this SFT-only run. See the "Why SFT-only?" note at the top.
Cell is kept as documentation; runs as a no-op.

The original 60-step / `beta=0.04` / 4-rollouts-per-prompt GRPOConfig is
preserved in git history and in the comment below for re-enablement once
the upstream version-skew between Unsloth's compiled cache and TRL is
resolved.


In [ ]:
# GRPO disabled in this SFT-only run.
#
# Reference config (the run we'd kick off if upstream version-skew were
# resolved):
#
#   from trl import GRPOTrainer, GRPOConfig
#   from training.grpo_reward import grpo_reward_fn, reset_history
#   reset_history()
#   grpo_args = GRPOConfig(
#       output_dir="/content/grpo_runs",
#       per_device_train_batch_size=GRPO_BATCH,
#       gradient_accumulation_steps=4,
#       num_generations=GRPO_NUM_GENERATIONS,
#       max_completion_length=160,
#       max_prompt_length=2048,
#       max_steps=GRPO_STEPS,         # 60 on A100/L40S
#       learning_rate=5e-6,
#       beta=0.04,                    # KL penalty
#       warmup_ratio=0.03,
#       lr_scheduler_type="cosine",
#       logging_steps=2,
#       save_strategy="no",
#       bf16=IS_A100, fp16=not IS_A100,
#       report_to="none",
#       seed=42,
#       gradient_checkpointing=True,
#       use_vllm=False,
#   )
#   GRPOTrainer(model=model, processing_class=tokenizer,
#               reward_funcs=grpo_reward_fn, args=grpo_args,
#               train_dataset=grpo_ds).train()
#   model.save_pretrained("/content/checkpoints/grpo_adapter")
print("[skip] Cell 8 — GRPO training disabled (SFT-only run).")


## Cell 9 — Final eval (random vs SFT)

**Expected runtime: 5–7 minutes on A100 / L40S, 12–15 minutes on T4.**
3 trained scenario families × 10 held-out seeds × 2 conditions = 60 episodes total.


In [ ]:
import time

model.eval()
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.max_length = None

# Trimmed to the 3 families covered by IDEAL_TRAJECTORIES (the SFT distribution).
# Evaluating on un-trained families would print 0% columns and look like failure.
EVAL_FAMILIES = [
    "oom_crash", "db_pool_exhaustion", "bad_deployment_cascade",
]
FINAL_SEEDS = list(range(8000, 8010))   # 10 held-out seeds
TOTAL_FINAL = len(EVAL_FAMILIES) * len(FINAL_SEEDS)

print(f"Final eval — {TOTAL_FINAL} episodes per condition × 2 conditions = {TOTAL_FINAL*2} total\n")

_t0 = time.monotonic()
print("→ random baseline")
report_random_final = evaluate("random", random_policy(rng_seed=12345),
                               EVAL_FAMILIES, FINAL_SEEDS,
                               system_prompt=SYSTEM_PROMPT, on_episode=_progress)
print(f"  random done in {time.monotonic()-_t0:.1f}s\n")

_t1 = time.monotonic()
print("→ SFT model")
report_sft_final = evaluate("sft", hf_pipeline_policy(hf_generate),
                            EVAL_FAMILIES, FINAL_SEEDS,
                            system_prompt=SYSTEM_PROMPT, on_episode=_progress)
print(f"  sft done in {time.monotonic()-_t1:.1f}s")

print("\n" + "=" * 70)
print("Final results (success rate per family):")
print("=" * 70)
print(f"{'family':<28s}  {'random':>8s}  {'sft':>8s}  {'delta':>8s}")
for fam in EVAL_FAMILIES:
    r_rate = report_random_final.by_family.get(fam, {}).get("success_rate", 0) * 100
    s_rate = report_sft_final.by_family.get(fam, {}).get("success_rate", 0) * 100
    delta = s_rate - r_rate
    sign = "+" if delta >= 0 else ""
    print(f"{fam:<28s}  {r_rate:>6.0f}%  {s_rate:>6.0f}%  {sign}{delta:>5.0f}pp")


## Cell 10 - Render the 4 canonical plots

**Expected runtime: < 1 minute.** Saves all four to `/content/results/` so
you can download them directly into the README.


In [ ]:
import time, json
from collections import defaultdict
from training.plots import (
    make_success_bars, make_action_distribution, save_figure,
)

_t0 = time.monotonic()
out_dir = "/content/results"

# 1) Success bars: random vs SFT across families
save_figure(
    make_success_bars(
        reports_by_condition={
            "random": report_random_final.by_family,
            "sft":    report_sft_final.by_family,
        },
        families=EVAL_FAMILIES,
    ),
    f"{out_dir}/final_success_rates.png",
)
print("  wrote final_success_rates.png")

# 2) Action distributions
def _actions(report):
    out = defaultdict(int)
    for ep in report.episodes:
        for a, _ in ep.actions:
            out[a] += 1
    return dict(out)

save_figure(
    make_action_distribution(actions_by_condition={
        "random": _actions(report_random_final),
        "sft":    _actions(report_sft_final),
    }),
    f"{out_dir}/final_action_distribution.png",
)
print("  wrote final_action_distribution.png")

# 3) Eval summary JSON for the README
summary = {
    "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct (4-bit nf4 + LoRA r=16)",
    "lora_r": 16,
    "lora_alpha": 32,
    "phase": "sft_only",
    "eval_seeds": FINAL_SEEDS,
    "by_family": {
        fam: {
            "random_success":   report_random_final.by_family[fam]["success_rate"],
            "sft_success":      report_sft_final.by_family[fam]["success_rate"],
            "random_avg_score": report_random_final.by_family[fam]["avg_score"],
            "sft_avg_score":    report_sft_final.by_family[fam]["avg_score"],
        }
        for fam in EVAL_FAMILIES
    },
}
with open(f"{out_dir}/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  wrote eval_summary.json")

print(f"\nAll plots written to {out_dir}/ in {time.monotonic()-_t0:.1f}s")
print("\nNote: SFT loss curve is captured by TRL's internal logging during cell 5.")
print("Download `/content/sft_runs/runs/` (TensorBoard logs) for the loss plot,")
print("or screenshot the per-step loss output from cell 5's progress bar.")


## Cell 11 - Push LoRA adapter to your HuggingFace Hub (optional)

**Expected runtime: 2–3 minutes.** Skipped if the `HF_USER` and `HF_TOKEN`
environment variables aren't set. Set them in Colab via:
> Sidebar → 🔑 Secrets → add `HF_TOKEN` (write scope) and `HF_USER`.


In [ ]:
import os, time
_t0 = time.monotonic()

HF_USER = os.environ.get("HF_USER", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_USER or not HF_TOKEN:
    print("HF_USER / HF_TOKEN not set — skipping Hub push.")
    print("To enable: Colab sidebar → 🔑 Secrets (or HF Space Settings → Variables and secrets)")
    print("           add HF_TOKEN (write scope) and HF_USER, then re-run this cell.")
else:
    from huggingface_hub import login, create_repo
    login(token=HF_TOKEN)
    repo_id = f"{HF_USER}/praetor-incident-commander-sft"
    create_repo(repo_id, exist_ok=True)
    model.push_to_hub(repo_id, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN)
    print(f"Pushed LoRA + tokenizer to https://huggingface.co/{repo_id}")
    print(f"  (paste this URL into the README's 'Trained LoRA adapter' line)")

print(f"\nDone in {time.monotonic()-_t0:.1f}s")


## Cell 12 - Print the README results table

**Expected runtime: < 1 second.** Copy this block straight into the
README's *Eval results* section, replacing the placeholder rows.


In [ ]:
print()
print("Paste the block below into README.md → 'Eval results' section:")
print()
print("=" * 70)
print()

# Build header dynamically from EVAL_FAMILIES so it stays in sync.
_pretty = {
    "oom_crash": "OOM Crash",
    "db_pool_exhaustion": "DB Pool",
    "bad_deployment_cascade": "Bad Deploy",
    "disk_full": "Disk Full",
    "slow_query": "Slow Query",
    "cert_expiry": "Cert Expiry",
}
_header_cells = [_pretty.get(f, f) for f in EVAL_FAMILIES]
print("| Condition | " + " | ".join(_header_cells) + " | Average |")
print("|---|" + "|".join(["---:"] * (len(EVAL_FAMILIES) + 1)) + "|")

def _row(name, by_fam, bold=False):
    rates = [by_fam.get(f, {}).get("success_rate", 0) * 100 for f in EVAL_FAMILIES]
    avg = sum(rates) / len(rates)
    label = f"**{name}**" if bold else name
    cells = " | ".join(f"{r:>3.0f}%" for r in rates)
    print(f"| {label} | {cells} | **{avg:>3.0f}%** |")

_n = len(FINAL_SEEDS) * len(EVAL_FAMILIES)
_row(f"Random (n={_n})", report_random_final.by_family)
_row(f"SFT (n={_n})", report_sft_final.by_family, bold=True)

print()
print("=" * 70)
print()
print("Plot files (commit these into results/ on GitHub):")
print("  /content/results/final_success_rates.png")
print("  /content/results/final_action_distribution.png")
print("  /content/results/eval_summary.json")
print()
print("Also screenshot cell 5's TRL progress bar for the SFT loss curve.")
